# Laboratório 10 — Pipeline Definitivo

**QLoRA (4-bit) + RAG massivo + KV Cache + FlashAttention-2**

Execução prevista no Google Colab Free (GPU T4, 15GB VRAM).

## Setup

Instalação das dependências e imports base.

In [ ]:
!pip install -q -U transformers bitsandbytes accelerate datasets sentencepiece matplotlib

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
assert torch.cuda.is_available(), "GPU CUDA é obrigatória (use Colab com runtime GPU)"
print("GPU:", torch.cuda.get_device_name(0))

## Passo 1 — Ingestão eficiente (QLoRA 4-bit)

Carregamento do modelo base em 4 bits via `bitsandbytes` para reduzir o footprint inicial de VRAM.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
)
model.eval()

In [ ]:
torch.cuda.synchronize()
vram_modelo_mb = torch.cuda.memory_allocated() / 1024**2
print(f"VRAM ocupada pelo modelo quantizado: {vram_modelo_mb:.1f} MB")

## Passo 2 — Simulando o RAG massivo

Em vez de invocar um pipeline de RAG completo, montamos o "contexto recuperado" concatenando abstracts reais do PubMed até atingir ~12.000 tokens — equivalente a 5 capítulos de manual médico recuperados pelo banco vetorial.

In [ ]:
from datasets import load_dataset

ds = load_dataset("pubmed_qa", "pqa_artificial", split="train", streaming=True)

trechos = []
total_chars = 0
ALVO_CARACTERES = 60_000

for row in ds:
    for paragrafo in row["context"]["contexts"]:
        trechos.append(paragrafo)
        total_chars += len(paragrafo)
        if total_chars >= ALVO_CARACTERES:
            break
    if total_chars >= ALVO_CARACTERES:
        break

contexto_massivo = "\n\n".join(trechos)
print(f"Trechos coletados: {len(trechos)}")
print(f"Caracteres totais: {len(contexto_massivo)}")

In [ ]:
PROMPT_SISTEMA = (
    "Você é um assistente médico. Com base estritamente nos trechos a seguir, "
    "redija um resumo clínico objetivo de 500 palavras destacando achados, "
    "metodologia e conclusões.\n\n"
)

prompt = PROMPT_SISTEMA + contexto_massivo + "\n\nResumo clínico:"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
n_tokens_prompt = inputs["input_ids"].shape[1]
print(f"Tokens no prompt: {n_tokens_prompt}")

## Passo 3 — O gargalo de geração (baseline sem KV Cache)

Geração de 100 tokens forçando `use_cache=False`. A cada token novo, todo o tensor Q, K, V do contexto é recalculado — complexidade O(n²) materializada de verdade. Instrumentamos tempo e pico de VRAM via `torch.cuda.max_memory_allocated`.

In [ ]:
import time

def bench_generate(model, inputs, n_novos_tokens=100, use_cache=True):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=n_novos_tokens,
            do_sample=False,
            use_cache=use_cache,
            pad_token_id=tokenizer.eos_token_id,
        )
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - t0
    peak_mb = torch.cuda.max_memory_allocated() / 1024**2
    return {"saida": out, "tempo_s": elapsed, "pico_vram_mb": peak_mb}

In [ ]:
model.config.use_cache = False

resultado_baseline = bench_generate(model, inputs, n_novos_tokens=100, use_cache=False)

print(f"[SEM KV Cache] Tempo:        {resultado_baseline['tempo_s']:.2f} s")
print(f"[SEM KV Cache] Pico de VRAM: {resultado_baseline['pico_vram_mb']:.1f} MB")